In [1]:
!nvidia-smi

Fri Mar  6 07:15:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   37C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q vllm lm-format-enforcer pandas datasets scikit-learn

In [8]:
# Need to install this version of protobuf after installing vllm
!pip install "protobuf==5.29.5"

  Using cached protobuf-5.29.5-cp38-abi3-manylinux2014_x86_64.whl.metadata (592 bytes)
Using cached protobuf-5.29.5-cp38-abi3-manylinux2014_x86_64.whl (319 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.5
    Uninstalling protobuf-6.33.5:
      Successfully uninstalled protobuf-6.33.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-reflection 1.78.0 requires protobuf<7.0.0,>=6.31.1, but you have protobuf 5.29.5 which is incompatible.
vllm 0.16.0 requires protobuf!=6.30.*,!=6.31.*,!=6.32.*,!=6.33.0.*,!=6.33.1.*,!=6.33.2.*,!=6.33.3.*,!=6.33.4.*,>=5.29.6, but you have protobuf 5.29.5 which is incompatible.


In [1]:
!pip install -q "datasets<4.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 14.5 MB/s eta 0:00:00


# RAFT Benchmark — 0-shot Evaluation with Qwen3-8B + vLLM

**RAFT** (Real-world Annotated Few-shot Tasks) is a benchmark of 11 real-world NLP classification tasks.
Each task provides exactly **50 labeled training examples**.
Test labels are withheld — official scoring requires submission to the leaderboard at https://huggingface.co/spaces/ought/raft-leaderboard.

### Available tasks
| Task key | Description |
|---|---|
| `ade_corpus_v2` | Adverse drug effects from medical text |
| `banking_77` | Customer service intent (77 classes) |
| `neurips_impact_statement_risks` | NeurIPS paper risk categorization |
| `one_stop_english` | Text readability level |
| `overruling` | Legal — is a rule overruled? |
| `semiconductor_org_types` | Semiconductor organization type |
| `systemic_functional_linguistics` | Clause type classification |
| `tai_safety_research` | AI safety paper categorization |
| `terms_of_service` | ToS clause policy violation |
| `tweet_eval_hate` | Hate speech detection |
| `twitter_complaints` | Twitter complaint detection |

**Strategy**: Since test labels are withheld, we:
1. Evaluate accuracy on the **50 training examples** (held-out fold)
2. Generate **test predictions** formatted for leaderboard submission


In [22]:
import os
import re
import json
import time
import numpy as np
from datasets import load_dataset
from tqdm import tqdm

# ── RAFT task metadata ─────────────────────────────────────────────────────────
# Each entry: task_key -> {text_fields, label_names, label_column}
RAFT_TASKS = {
    "ade_corpus_v2": {
        "text_fields": ["Sentence"],
        "label_names": ["not ADE-related", "ADE-related"],   # labels 1, 2
        "description": "Classify whether the sentence describes an Adverse Drug Effect (ADE).",
    },
    "banking_77": {
        "text_fields": ["Query"],
        "label_names": [
            "activate my card", "age limit", "apple pay or google pay", "atm support",
            "automatic top up", "balance not updated after bank transfer",
            "balance not updated after cheque or cash deposit", "beneficiary not allowed",
            "cancel transfer", "card about to expire", "card acceptance",
            "card arrival", "card delivery estimate", "card linking",
            "card not working", "card payment fee charged", "card payment not recognised",
            "card payment wrong exchange rate", "card swallowed", "cash withdrawal charge",
            "cash withdrawal not recognised", "change pin", "compromised card",
            "contactless not working", "country support", "declined card payment",
            "declined cash withdrawal", "declined transfer", "direct debit payment not recognised",
            "disposable card limits", "edit personal details", "exchange charge",
            "exchange rate", "exchange via app", "extra charge on statement",
            "failed transfer", "fiat currency support", "get disposable virtual card",
            "get physical card", "getting spare card", "getting virtual card",
            "lost or stolen card", "lost or stolen phone", "order physical card",
            "passcode forgotten", "pending card payment", "pending cash withdrawal",
            "pending top up", "pending transfer", "pin blocked", "receiving money",
            "refund not showing up", "request refund", "reverted card payment?",
            "supported cards and currencies", "terminate account", "top up by bank transfer or wire",
            "top up by card charge", "top up by cash or cheque", "top up failed",
            "top up limits", "top up reverted", "topping up by card",
            "transaction charged twice", "transfer fee charged", "transfer into account",
            "transfer not received by recipient", "transfer timing",
            "unable to verify identity", "verify my identity", "verify source of funds",
            "verify top up", "virtual card not working", "visa or mastercard",
            "why verify identity", "wrong amount of cash received", "wrong exchange rate for cash withdrawal",
        ],   # labels 1–77
        "description": "Classify the customer service query into one of 77 banking intent categories.",
    },
    "neurips_impact_statement_risks": {
        "text_fields": ["Paper title", "Impact statement"],
        "label_names": ["doesn't mention a harmful application", "mentions a harmful application"],
        "description": "Does the NeurIPS impact statement mention a harmful application of the research?",
    },
    "one_stop_english": {
        "text_fields": ["Article"],
        "label_names": ["elementary", "intermediate", "advanced"],
        "description": "Classify the reading level of the article: elementary, intermediate, or advanced.",
    },
    "overruling": {
        "text_fields": ["Sentence"],
        "label_names": ["not overruling", "overruling"],
        "description": "Does this legal sentence overrule a previous legal rule or case?",
    },
    "semiconductor_org_types": {
        "text_fields": ["Organization name", "Description"],
        "label_names": [
            "university", "company", "research institute",
            "other", "government",
        ],
        "description": "Classify the semiconductor organization type.",
    },
    "tai_safety_research": {
        "text_fields": ["Title", "Abstract Note"],
        "label_names": ["not TAI safety research", "TAI safety research"],
        "description": "Is this paper related to transformative AI (TAI) safety research?",
    },
    "terms_of_service": {
        "text_fields": ["Sentence"],
        "label_names": ["not potentially unfair", "potentially unfair"],
        "description": "Does this Terms of Service clause contain a potentially unfair policy?",
    },
    "tweet_eval_hate": {
        "text_fields": ["Tweet"],
        "label_names": ["not hate speech", "hate speech"],
        "description": "Does this tweet contain hate speech?",
    },
    "twitter_complaints": {
        "text_fields": ["Tweet text"],
        "label_names": ["not a complaint", "complaint"],
        "description": "Is this tweet a customer complaint?",
    },
}

print(f"RAFT benchmark: {len(RAFT_TASKS)} tasks available")
for task, meta in RAFT_TASKS.items():
    n_labels = len(meta["label_names"])
    print(f"  {task}: {n_labels} label(s), input field(s): {meta['text_fields']}")

RAFT benchmark: 10 tasks available
  ade_corpus_v2: 2 label(s), input field(s): ['Sentence']
  banking_77: 77 label(s), input field(s): ['Query']
  neurips_impact_statement_risks: 2 label(s), input field(s): ['Paper title', 'Impact statement']
  one_stop_english: 3 label(s), input field(s): ['Article']
  overruling: 2 label(s), input field(s): ['Sentence']
  semiconductor_org_types: 5 label(s), input field(s): ['Organization name', 'Description']
  tai_safety_research: 2 label(s), input field(s): ['Title', 'Abstract Note']
  terms_of_service: 2 label(s), input field(s): ['Sentence']
  tweet_eval_hate: 2 label(s), input field(s): ['Tweet']
  twitter_complaints: 2 label(s), input field(s): ['Tweet text']


In [23]:
# ── Select task(s) to evaluate ────────────────────────────────────────────────
# Set TASKS_TO_EVAL to a list of task keys, or use ALL_TASKS to run everything.
# Note: banking_77 has 77 classes and may be harder for 0-shot; start with easier tasks.

ALL_TASKS = list(RAFT_TASKS.keys())

TASKS_TO_EVAL = ALL_TASKS # Edit this list, or set TASKS_TO_EVAL = ALL_TASKS to run all 11 tasks

print(f"Tasks to evaluate ({len(TASKS_TO_EVAL)}): {TASKS_TO_EVAL}")

Tasks to evaluate (10): ['ade_corpus_v2', 'banking_77', 'neurips_impact_statement_risks', 'one_stop_english', 'overruling', 'semiconductor_org_types', 'tai_safety_research', 'terms_of_service', 'tweet_eval_hate', 'twitter_complaints']


In [24]:
# ── Load datasets ─────────────────────────────────────────────────────────────
# RAFT splits: 'train' (50 labeled examples), 'test' (unlabeled, labels withheld)
# We evaluate accuracy on 'train' and generate predictions for 'test'.

task_datasets = {}   # task -> {"train": Dataset, "test": Dataset}

for task in TASKS_TO_EVAL:
    print(f"Loading {task}...")
    train_ds = load_dataset("ought/raft", task, split="train")
    test_ds  = load_dataset("ought/raft", task, split="test")
    task_datasets[task] = {"train": train_ds, "test": test_ds}
    print(f"  train: {len(train_ds)} examples | test: {len(test_ds)} examples")
    print(f"  columns: {train_ds.column_names}")
    # RAFT uses integer labels starting at 1 (0 = unlabeled/unknown in some tasks)
    label_vals = sorted(set(train_ds["Label"]))
    print(f"  train label values: {label_vals}")

Loading ade_corpus_v2...
  train: 50 examples | test: 5000 examples
  columns: ['Sentence', 'ID', 'Label']
  train label values: [1, 2]
Loading banking_77...
  train: 50 examples | test: 5000 examples
  columns: ['Query', 'ID', 'Label']
  train label values: [6, 7, 8, 9, 13, 14, 15, 17, 19, 21, 22, 23, 26, 27, 29, 30, 31, 32, 34, 36, 40, 41, 47, 49, 50, 53, 54, 55, 58, 60, 61, 70, 71, 74, 75, 76, 77]
Loading neurips_impact_statement_risks...
  train: 50 examples | test: 150 examples
  columns: ['Paper title', 'Paper link', 'Impact statement', 'ID', 'Label']
  train label values: [1, 2]
Loading one_stop_english...
  train: 50 examples | test: 516 examples
  columns: ['Article', 'ID', 'Label']
  train label values: [1, 2, 3]
Loading overruling...
  train: 50 examples | test: 2350 examples
  columns: ['Sentence', 'ID', 'Label']
  train label values: [1, 2]
Loading semiconductor_org_types...
  train: 50 examples | test: 449 examples
  columns: ['Paper title', 'Organization name', 'ID', 'La

0000.parquet:   0%|          | 0.00/38.5k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/912k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1639 [00:00<?, ? examples/s]

  train: 50 examples | test: 1639 examples
  columns: ['Title', 'Abstract Note', 'Url', 'Publication Year', 'Item Type', 'Author', 'Publication Title', 'ID', 'Label']
  train label values: [1, 2]
Loading terms_of_service...
  train: 50 examples | test: 5000 examples
  columns: ['Sentence', 'ID', 'Label']
  train label values: [1, 2]
Loading tweet_eval_hate...
  train: 50 examples | test: 2966 examples
  columns: ['Tweet', 'ID', 'Label']
  train label values: [1, 2]
Loading twitter_complaints...
  train: 50 examples | test: 3399 examples
  columns: ['Tweet text', 'ID', 'Label']
  train label values: [1, 2]


In [6]:
# ── Config ────────────────────────────────────────────────────────────────────
MODEL_NAME      = "Qwen/Qwen3-8B"
MAX_NEW_TOKENS  = 32          # RAFT labels are short; 32 is sufficient
ENABLE_THINKING = False       # Set True to enable chain-of-thought (slower)

print({
    "model": MODEL_NAME,
    "max_new_tokens": MAX_NEW_TOKENS,
    "enable_thinking": ENABLE_THINKING,
    "tasks": TASKS_TO_EVAL,
})

{'model': 'Qwen/Qwen3-8B', 'max_new_tokens': 32, 'enable_thinking': False, 'tasks': ['ade_corpus_v2', 'overruling', 'tweet_eval_hate', 'twitter_complaints', 'terms_of_service']}


In [13]:
import torch
cuda_ver = torch.version.cuda.replace(".", "")  # e.g. "121"
torch_ver = torch.__version__.split("+")[0]      # e.g. "2.3.0"
print(f"PyTorch {torch_ver}, CUDA {cuda_ver}")

!pip uninstall vllm -y
!pip install vllm --extra-index-url https://download.pytorch.org/whl/cu{cuda_ver}

PyTorch 2.10.0, CUDA 128
Found existing installation: vllm 0.16.0
Uninstalling vllm-0.16.0:
  Successfully uninstalled vllm-0.16.0
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu128
  Using cached vllm-0.16.0-cp38-abi3-manylinux_2_31_x86_64.whl.metadata (9.4 kB)
  Using cached protobuf-6.33.5-cp39-abi3-manylinux2014_x86_64.whl.metadata (593 bytes)
Using cached vllm-0.16.0-cp38-abi3-manylinux_2_31_x86_64.whl (508.3 MB)
Using cached protobuf-6.33.5-cp39-abi3-manylinux2014_x86_64.whl (323 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.5 which is incompatible.
tensorflow 2.19.0 requires

In [9]:
# ── Load model ────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams

    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

    print("Loading Qwen3-8B with vLLM...")
    llm = LLM(
        model=MODEL_NAME,
        dtype="float16",
        gpu_memory_utilization=0.95,
        max_model_len=4096,
        enforce_eager=False,
    )

    sampling_params = SamplingParams(
        temperature=0,
        top_k=20,
        max_tokens=MAX_NEW_TOKENS,
    )

    print("Model loaded!")

Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Loading Qwen3-8B with vLLM...
INFO 03-06 07:16:29 [utils.py:223] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.95, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-8B'}


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

INFO 03-06 07:16:51 [model.py:529] Resolved architecture: Qwen3ForCausalLM
WARNING 03-06 07:16:51 [model.py:1874] Casting torch.bfloat16 to torch.float16.
INFO 03-06 07:16:51 [model.py:1549] Using max model len 4096
INFO 03-06 07:16:51 [scheduler.py:224] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-06 07:16:51 [vllm.py:689] Asynchronous scheduling is enabled.


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

WARNING 03-06 07:16:52 [system_utils.py:140] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 03-06 07:19:27 [llm.py:355] Supported tasks: ['generate']
Model loaded!


In [10]:
# ── Output directory ──────────────────────────────────────────────────────────
IN_COLAB = True
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE_SAVE_DIR = "/content/drive/MyDrive/Colab_Data/RAFT/Qwen3_8B_RAFT_Eval_vLLM"
else:
    DRIVE_SAVE_DIR = os.path.abspath("./qwen3_8b_raft_eval_vllm_outputs")

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"Saving results to: {DRIVE_SAVE_DIR}")

Mounted at /content/drive
Saving results to: /content/drive/MyDrive/Colab_Data/RAFT/Qwen3_8B_RAFT_Eval_vLLM


In [25]:
# ── Prompt builder ─────────────────────────────────────────────────────────────
def build_system_prompt(task: str) -> str:
    meta = RAFT_TASKS[task]
    label_list = ", ".join(f'"{l}"' for l in meta["label_names"])
    return (
        f"You are a text classification assistant. "
        f"{meta['description']} "
        f"Answer with exactly one of the following labels: {label_list}. "
        f"Output only the label, nothing else."
    )


def build_user_content(task: str, example: dict) -> str:
    """Format the example fields as the user message."""
    meta = RAFT_TASKS[task]
    parts = []
    for field in meta["text_fields"]:
        value = example.get(field, "").strip()
        if value:
            parts.append(f"{field}: {value}")
    parts.append("Label:")
    return "\n".join(parts)


def build_prompt(task: str, example: dict) -> str:
    messages = [
        {"role": "system", "content": build_system_prompt(task)},
        {"role": "user",   "content": build_user_content(task, example)},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,
    )


# Quick sanity check
sample_task = TASKS_TO_EVAL[0]
sample_ex   = task_datasets[sample_task]["train"][0]
print(f"== Example prompt for task '{sample_task}' ==")
print(build_prompt(sample_task, sample_ex)[:800])

== Example prompt for task 'ade_corpus_v2' ==
<|im_start|>system
You are a text classification assistant. Classify whether the sentence describes an Adverse Drug Effect (ADE). Answer with exactly one of the following labels: "not ADE-related", "ADE-related". Output only the label, nothing else.<|im_end|>
<|im_start|>user
Sentence: No regional side effects were noted.
Label:<|im_end|>
<|im_start|>assistant
<think>

</think>




In [26]:
# ── Label extractor ───────────────────────────────────────────────────────────
def strip_thinking(text: str) -> str:
    if "</think>" in text:
        return text.split("</think>", 1)[-1].strip()
    return text


def extract_label(task: str, response: str):
    """
    Map raw model response to a RAFT integer label (1-indexed).
    Returns None if no match found.
    RAFT labels are 1-indexed: label_names[0] -> label 1, label_names[1] -> label 2, etc.
    """
    cleaned = strip_thinking(response).strip().lower()
    label_names = RAFT_TASKS[task]["label_names"]

    # Try exact / substring match, longest-first to avoid partial matches
    sorted_labels = sorted(enumerate(label_names), key=lambda x: -len(x[1]))
    for idx, name in sorted_labels:
        if name.lower() in cleaned:
            return idx + 1  # RAFT is 1-indexed
    return None


# Test extraction
print(extract_label("ade_corpus_v2", "ADE-related"))          # expect 2
print(extract_label("ade_corpus_v2", "not ADE-related"))      # expect 1
print(extract_label("tweet_eval_hate", "not hate speech"))    # expect 1
print(extract_label("tweet_eval_hate", "hate speech"))        # expect 2

2
1
1
2


In [27]:
# ── Generate & evaluate — all selected tasks ─────────────────────────────────
from sklearn.metrics import accuracy_score, classification_report

all_task_results = {}   # task -> {"train_results": [...], "test_predictions": [...]}
summary_rows = []

for task in TASKS_TO_EVAL:
    print(f"\n{'='*60}")
    print(f"Task: {task}")
    print(f"{'='*60}")

    train_ds = task_datasets[task]["train"]
    test_ds  = task_datasets[task]["test"]
    meta     = RAFT_TASKS[task]

    # ── Build prompts for train (for accuracy) and test (for submission) ──────
    train_prompts = [build_prompt(task, ex) for ex in train_ds]
    test_prompts  = [build_prompt(task, ex) for ex in test_ds]
    all_prompts   = train_prompts + test_prompts

    print(f"  {len(train_prompts)} train prompts + {len(test_prompts)} test prompts = {len(all_prompts)} total")

    # ── Generate responses ─────────────────────────────────────────────────────
    t0 = time.time()
    outputs = llm.generate(all_prompts, sampling_params)
    gen_time   = time.time() - t0
    total_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
    throughput   = total_tokens / gen_time if gen_time > 0 else 0.0
    print(f"  Time:       {gen_time/60:.1f} min")
    print(f"  Tokens:     {total_tokens:,}")
    print(f"  Throughput: {throughput:.1f} tokens/sec")

    # ── Score train predictions ────────────────────────────────────────────────
    train_results = []
    for i, (ex, out) in enumerate(zip(train_ds, outputs[:len(train_prompts)])):
        response = out.outputs[0].text
        gt       = int(ex["Label"])          # true label (1-indexed)
        pred     = extract_label(task, response)
        train_results.append({
            "id":           ex.get("ID", i),
            "ground_truth": gt,
            "prediction":   pred,
            "raw_response": response.strip(),
            "is_correct":   (pred == gt) if pred is not None else False,
            "is_unknown":   pred is None,
        })

    correct  = sum(1 for r in train_results if r["is_correct"])
    unknown  = sum(1 for r in train_results if r["is_unknown"])
    accuracy = correct / len(train_results)

    # Accuracy excluding unknowns
    known_pairs = [(r["ground_truth"], r["prediction"])
                   for r in train_results if r["prediction"] is not None]
    acc_known = accuracy_score(
        [p[0] for p in known_pairs], [p[1] for p in known_pairs]
    ) if known_pairs else 0.0

    print(f"\n  Train accuracy (n={len(train_results)}): {accuracy:.4f}  "
          f"(known-only: {acc_known:.4f}, unknown: {unknown})")

    # Classification report
    gt_list   = [r["ground_truth"] for r in train_results if r["prediction"] is not None]
    pred_list = [r["prediction"]   for r in train_results if r["prediction"] is not None]
    if pred_list:
        label_ids   = list(range(1, len(meta["label_names"]) + 1))
        label_names = meta["label_names"]
        # Only report classes that appear in ground truth or predictions
        present = sorted(set(gt_list) | set(pred_list))
        present_names = [label_names[p - 1] for p in present]
        print(classification_report(gt_list, pred_list,
                                     labels=present,
                                     target_names=present_names))

    # ── Collect test predictions (for leaderboard submission) ──────────────────
    test_predictions = []
    for i, (ex, out) in enumerate(zip(test_ds, outputs[len(train_prompts):])):
        response = out.outputs[0].text
        pred     = extract_label(task, response)
        # RAFT submission: if we can't parse, default to label 1 (most common)
        pred_safe = pred if pred is not None else 1
        test_predictions.append({
            "ID":           ex.get("ID", i),
            "Label":        pred_safe,
            "raw_response": response.strip(),
        })

    # ── Save per-task results ──────────────────────────────────────────────────
    task_dir = os.path.join(DRIVE_SAVE_DIR, task)
    os.makedirs(task_dir, exist_ok=True)

    with open(os.path.join(task_dir, "train_results.json"), "w") as f:
        json.dump(train_results, f, indent=2)

    # Submission CSV: ID + Label columns only (RAFT format)
    import csv
    submission_path = os.path.join(task_dir, "test_predictions.csv")
    with open(submission_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["ID", "Label"])
        writer.writeheader()
        for row in test_predictions:
            writer.writerow({"ID": row["ID"], "Label": row["Label"]})

    print(f"  Test predictions saved: {submission_path}")

    # Store for summary
    all_task_results[task] = {
        "train_results":    train_results,
        "test_predictions": test_predictions,
    }
    summary_rows.append({
        "task":              task,
        "n_train":           len(train_results),
        "n_test":            len(test_predictions),
        "accuracy":          round(accuracy, 4),
        "accuracy_known":    round(acc_known, 4),
        "unknown_frac":      round(unknown / len(train_results), 4),
        "n_labels":          len(meta["label_names"]),
        "total_new_tokens":  total_tokens,
        "throughput_tok_per_sec": round(throughput, 1),
        "gen_time_min":      round(gen_time / 60, 2),
    })


Task: ade_corpus_v2
  50 train prompts + 5000 test prompts = 5050 total


Adding requests:   0%|          | 0/5050 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/5050 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

  Time:       1.4 min
  Tokens:     20,660
  Throughput: 238.1 tokens/sec

  Train accuracy (n=50): 0.1400  (known-only: 0.1400, unknown: 0)
                 precision    recall  f1-score   support

not ADE-related       0.00      0.00      0.00        15
    ADE-related       0.32      0.20      0.25        35

       accuracy                           0.14        50
      macro avg       0.16      0.10      0.12        50
   weighted avg       0.22      0.14      0.17        50

  Test predictions saved: /content/drive/MyDrive/Colab_Data/RAFT/Qwen3_8B_RAFT_Eval_vLLM/ade_corpus_v2/test_predictions.csv

Task: banking_77
  50 train prompts + 5000 test prompts = 5050 total


Adding requests:   0%|          | 0/5050 [00:00<?, ?it/s]

...

Processed prompts:   0%|          | 0/3016 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

  Time:       0.9 min
  Tokens:     12,064
  Throughput: 230.3 tokens/sec

  Train accuracy (n=50): 0.2600  (known-only: 0.2600, unknown: 0)
                 precision    recall  f1-score   support

not hate speech       0.21      0.29      0.24        21
    hate speech       0.32      0.24      0.27        29

       accuracy                           0.26        50
      macro avg       0.27      0.26      0.26        50
   weighted avg       0.27      0.26      0.26        50

  Test predictions saved: /content/drive/MyDrive/Colab_Data/RAFT/Qwen3_8B_RAFT_Eval_vLLM/tweet_eval_hate/test_predictions.csv

Task: twitter_complaints
  50 train prompts + 3399 test prompts = 3449 total


Adding requests:   0%|          | 0/3449 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/3449 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

  Time:       0.8 min
  Tokens:     12,309
  Throughput: 244.7 tokens/sec

  Train accuracy (n=50): 0.1200  (known-only: 0.1200, unknown: 0)
                 precision    recall  f1-score   support

not a complaint       0.03      0.06      0.04        17
      complaint       0.24      0.15      0.19        33

       accuracy                           0.12        50
      macro avg       0.14      0.11      0.11        50
   weighted avg       0.17      0.12      0.14        50

  Test predictions saved: /content/drive/MyDrive/Colab_Data/RAFT/Qwen3_8B_RAFT_Eval_vLLM/twitter_complaints/test_predictions.csv


In [28]:
# ── Final summary across tasks ─────────────────────────────────────────────────
import pandas as pd

summary_df = pd.DataFrame(summary_rows)

print("\n" + "="*70)
print("RAFT 0-SHOT EVALUATION SUMMARY")
print(f"Model: {MODEL_NAME}  |  enable_thinking={ENABLE_THINKING}")
print("="*70)
print(summary_df.to_string(index=False))

if len(summary_rows) > 1:
    avg_acc = summary_df["accuracy"].mean()
    avg_acc_known = summary_df["accuracy_known"].mean()
    print(f"\nMean accuracy (all tasks):    {avg_acc:.4f}")
    print(f"Mean accuracy (known only):   {avg_acc_known:.4f}")

# Save summary
SUMMARY_FILE = os.path.join(DRIVE_SAVE_DIR, "raft_summary.json")
final_summary = {
    "model":           MODEL_NAME,
    "enable_thinking": ENABLE_THINKING,
    "tasks":           TASKS_TO_EVAL,
    "per_task":        summary_rows,
    "mean_accuracy":   float(summary_df["accuracy"].mean()) if len(summary_rows) > 1 else summary_rows[0]["accuracy"],
}
with open(SUMMARY_FILE, "w") as f:
    json.dump(final_summary, f, indent=2)
print(f"\nSummary saved to: {SUMMARY_FILE}")

print("\nNote: Test predictions (test_predictions.csv per task) can be submitted")
print("to the RAFT leaderboard: https://huggingface.co/spaces/ought/raft-leaderboard")


RAFT 0-SHOT EVALUATION SUMMARY
Model: Qwen/Qwen3-8B  |  enable_thinking=False
                          task  n_train  n_test  accuracy  accuracy_known  unknown_frac  n_labels  total_new_tokens  throughput_tok_per_sec  gen_time_min
                 ade_corpus_v2       50    5000      0.14          0.1400          0.00         2             20660                   238.1          1.45
                    banking_77       50    5000      0.24          0.2449          0.02        77             22244                   291.4          1.27
neurips_impact_statement_risks       50     150      0.90          0.9000          0.00         2              1252                    81.2          0.26
              one_stop_english       50     516      0.40          0.4000          0.00         3              1617                    10.4          2.59
                    overruling       50    2350      0.96          0.9600          0.00         2             10844                   203.2          0.

In [29]:
# ── (Optional) Combine test_predictions.csv files into a single submission ───
# The official RAFT submission expects one CSV per task.
# This cell just lists what was generated.

print("Generated submission files:")
for task in TASKS_TO_EVAL:
    task_dir = os.path.join(DRIVE_SAVE_DIR, task)
    csv_path = os.path.join(task_dir, "test_predictions.csv")
    if os.path.exists(csv_path):
        import pandas as pd
        df = pd.read_csv(csv_path)
        print(f"  {task}: {len(df)} rows  →  {csv_path}")

Generated submission files:
  ade_corpus_v2: 5000 rows  →  /content/drive/MyDrive/Colab_Data/RAFT/Qwen3_8B_RAFT_Eval_vLLM/ade_corpus_v2/test_predictions.csv
  banking_77: 5000 rows  →  /content/drive/MyDrive/Colab_Data/RAFT/Qwen3_8B_RAFT_Eval_vLLM/banking_77/test_predictions.csv
  neurips_impact_statement_risks: 150 rows  →  /content/drive/MyDrive/Colab_Data/RAFT/Qwen3_8B_RAFT_Eval_vLLM/neurips_impact_statement_risks/test_predictions.csv
  one_stop_english: 516 rows  →  /content/drive/MyDrive/Colab_Data/RAFT/Qwen3_8B_RAFT_Eval_vLLM/one_stop_english/test_predictions.csv
  overruling: 2350 rows  →  /content/drive/MyDrive/Colab_Data/RAFT/Qwen3_8B_RAFT_Eval_vLLM/overruling/test_predictions.csv
  semiconductor_org_types: 449 rows  →  /content/drive/MyDrive/Colab_Data/RAFT/Qwen3_8B_RAFT_Eval_vLLM/semiconductor_org_types/test_predictions.csv
  tai_safety_research: 1639 rows  →  /content/drive/MyDrive/Colab_Data/RAFT/Qwen3_8B_RAFT_Eval_vLLM/tai_safety_research/test_predictions.csv
  terms_of_s